In [1]:
#import and settings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests

# Must match the timezone and preprocessing used during training.
TIMEZONE = "Europe/Berlin"
ROUND_BEFORE_FEATURES = True

LATITUDE = 50.08
LONGITUDE = 7.76

WATER_API = "https://www.pegelonline.wsv.de/webservices/rest-api/v2"
WEATHER_API = "https://api.open-meteo.com/v1/forecast"


def today_local():
    return (
        pd.Timestamp.now(tz=TIMEZONE)
        .normalize()
        .tz_localize(None)
    )


def get_json(url, params=None):
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

In [2]:
#fetch water data
def fetch_water_data():
    stations = get_json(f"{WATER_API}/stations.json")

    station = next(
        (
            item for item in stations
            if item.get("shortname", "").strip().upper() == "KAUB"
        ),
        None,
    )

    if station is None:
        raise ValueError("Kaub station was not found.")

    measurements = get_json(
        f"{WATER_API}/stations/{station['uuid']}/W/measurements.json",
        params={"start": "P30D"},
    )

    water = pd.DataFrame(measurements)

    if water.empty:
        raise ValueError("No water measurements returned for Kaub.")

    return water

In [3]:
#clean water data
def clean_water_data(water):
    if not {"timestamp", "value"}.issubset(water.columns):
        raise ValueError("Water data needs timestamp and value columns.")

    water = water.copy()

    water["timestamp"] = pd.to_datetime(
        water["timestamp"], utc=True, errors="coerce"
    )
    water["value"] = pd.to_numeric(
        water["value"], errors="coerce"
    )

    water = water.dropna(subset=["timestamp", "value"])

    if water.empty:
        raise ValueError("No valid water measurements remain.")

    water = (
        water.groupby("timestamp")[["value"]]
        .mean()
        .sort_index()
    )

    water.index = water.index.tz_convert(TIMEZONE)

    return water.rename(columns={"value": "water_level_mean"})

In [4]:
#Aggregate water data to daily data
def create_daily_water(water):
    # First hourly means, then daily means.
    hourly = water["water_level_mean"].resample("h").mean()

    daily = hourly.resample("D").agg(["mean", "count"])
    daily.columns = ["water_level_mean", "hours_available"]

    # Account for 23-hour and 25-hour days during clock changes.
    expected_hours = pd.Series(
        [
            len(
                pd.date_range(
                    start=day,
                    end=day + pd.DateOffset(days=1),
                    freq="h",
                    inclusive="left",
                )
            )
            for day in daily.index
        ],
        index=daily.index,
    )

    incomplete = daily["hours_available"] < expected_hours
    daily.loc[incomplete, "water_level_mean"] = np.nan

    daily.index = daily.index.tz_localize(None)
    daily.index.name = "date"

    # Exclude today's unfinished data.
    return daily.loc[daily.index < today_local()]

In [5]:
#fetch weather data
def fetch_weather_data():
    payload = get_json(
        WEATHER_API,
        params={
            "latitude": LATITUDE,
            "longitude": LONGITUDE,
            "daily": "temperature_2m_mean,precipitation_sum",
            "timezone": TIMEZONE,
            "past_days": 35,
            "forecast_days": 1,
        },
    )

    daily = payload.get("daily")

    if not daily or not daily.get("time"):
        raise ValueError("No daily weather data returned.")

    return pd.DataFrame(daily)

In [6]:
#clean weather data
def clean_weather_data(weather):
    required = {"time", "temperature_2m_mean", "precipitation_sum"}

    if not required.issubset(weather.columns):
        raise ValueError("Weather data is missing required columns.")

    weather = weather.rename(
        columns={
            "time": "date",
            "temperature_2m_mean": "temperature_mean",
        }
    ).copy()

    weather["date"] = pd.to_datetime(
        weather["date"], errors="coerce"
    )

    for column in ["temperature_mean", "precipitation_sum"]:
        weather[column] = pd.to_numeric(
            weather[column], errors="coerce"
        )

    weather = (
        weather.dropna(subset=["date"])
        .set_index("date")
        .sort_index()
    )

    if weather.index.has_duplicates:
        raise ValueError("Duplicate dates in weather data.")

    weather = weather.loc[
        weather.index < today_local(),
        ["temperature_mean", "precipitation_sum"],
    ]

    if weather.empty:
        raise ValueError("No completed weather days available.")

    return weather

In [7]:
#merge water and weather
def merge_live_data(water, weather):
    # Keep the longer weather history for the 30-day rainfall feature.
    realtime = water.join(weather, how="outer").sort_index()

    # Missing dates stay visible rather than shifting lag meanings.
    realtime = realtime.asfreq("D")
    realtime.index.name = "date"

    return realtime.loc[realtime.index < today_local()]

In [8]:
#validate the merged data
def validate_live_data(realtime):
    if realtime.empty:
        raise ValueError("Merged live data is empty.")

    if not isinstance(realtime.index, pd.DatetimeIndex):
        raise ValueError("Live data must have a DatetimeIndex.")

    if realtime.index.has_duplicates:
        raise ValueError("Duplicate dates in live data.")

    required = [
        "water_level_mean",
        "temperature_mean",
        "precipitation_sum",
    ]

    missing = [
        column for column in required
        if column not in realtime.columns
    ]

    if missing:
        raise ValueError(f"Missing live columns: {missing}")

    yesterday = today_local() - pd.Timedelta(days=1)

    if realtime.index[-1] != yesterday:
        raise ValueError(
            f"Latest date is {realtime.index[-1].date()}; "
            f"expected {yesterday.date()}."
        )

    if realtime.loc[yesterday, required].isna().any():
        raise ValueError(
            f"Water or weather data is incomplete for {yesterday.date()}."
        )

In [9]:
#create model features
def create_live_features(realtime):
    df = realtime.sort_index().copy()

    columns = [
        "water_level_mean",
        "precipitation_sum",
        "temperature_mean",
    ]

    if ROUND_BEFORE_FEATURES:
        df[columns] = df[columns].round()

    features = pd.DataFrame(index=df.index)

    for lag in [0, 1, 2, 3, 7, 14, 28]:
        features[f"lag_{lag}"] = df["water_level_mean"].shift(lag)

    for days in [3, 7, 14, 30]:
        features[f"precip_{days}d"] = (
            df["precipitation_sum"]
            .rolling(days, min_periods=days)
            .sum()
        )

    for days in [3, 7]:
        features[f"temp_{days}d"] = (
            df["temperature_mean"]
            .rolling(days, min_periods=days)
            .mean()
        )

    # Do not drop rows: we must check the latest date explicitly.
    return features

In [10]:
def realtime_pipeline(feature_names):
    water = fetch_water_data()
    water = clean_water_data(water)
    water = create_daily_water(water)

    weather = fetch_weather_data()
    weather = clean_weather_data(weather)

    realtime = merge_live_data(water, weather)
    validate_live_data(realtime)

    live_features = create_live_features(realtime)

    if live_features.empty:
        raise ValueError("No live features available.")

    missing = [
        name for name in feature_names
        if name not in live_features.columns
    ]

    if missing:
        raise ValueError(f"Missing model features: {missing}")

    X_live = live_features.loc[:, feature_names].iloc[[-1]]

    invalid = ~np.isfinite(X_live.to_numpy(dtype=float))

    if invalid.any():
        bad_columns = X_live.columns[invalid[0]].tolist()
        raise ValueError(
            f"Incomplete history or invalid values for: {bad_columns}"
        )

    return realtime, X_live

In [11]:
# Search the current directory and its parents for the model.
model_relative_path = Path("models") / "water_level_model.joblib"

MODEL_PATH = next(
    (
        folder / model_relative_path
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / model_relative_path).is_file()
    ),
    None,
)

if MODEL_PATH is None:
    raise FileNotFoundError(
        "Could not find models/water_level_model.joblib. "
        "Check your model filename and notebook location."
    )

saved = joblib.load(MODEL_PATH)

model = saved["model"]
feature_names = list(saved["features"])
horizon = int(saved["horizon"])

print("Model loaded:", MODEL_PATH)
print("Forecast horizon:", horizon, "days")
print("Features:", feature_names)

Model loaded: c:\Users\Prithvi Hegde\neuefische_course\Rhine-cargo-advisor\models\water_level_model.joblib
Forecast horizon: 7 days
Features: ['lag_0', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'precip_3d', 'precip_7d', 'precip_14d', 'precip_30d', 'temp_3d', 'temp_7d']


In [12]:
#Run the live pipeline and inspect inputs
realtime, X_live = realtime_pipeline(feature_names)

display(realtime.tail())
display(X_live)

,water_level_mean,hours_available,temperature_mean,precipitation_sum
date,,,,
2026-10-06,1.583333,24.0,14.2,0.0
2026-10-07,1.322917,24.0,15.4,1.0
2026-10-08,0.989583,24.0,13.5,9.1
2026-10-09,7.583333,24.0,9.5,1.8
2026-10-10,7.822917,24.0,12.9,2.3


,lag_0,lag_1,lag_2,lag_3,lag_7,lag_14,lag_28,precip_3d,precip_7d,precip_14d,precip_30d,temp_3d,temp_7d
date,,,,,,,,,,,,,
2026-10-10,8.0,8.0,1.0,1.0,8.0,-1.0,21.0,13.0,14.0,28.0,33.0,12.333333,13.571429


In [14]:
#generate the forecast
prediction = float(model.predict(X_live)[0])

if not np.isfinite(prediction):
    raise ValueError("The model returned an invalid prediction.")

reference_date = pd.Timestamp(X_live.index[0])
forecast_date = reference_date + pd.Timedelta(days=horizon)

print(f"Latest observation: {reference_date.date()}")
print(f"Forecast date:      {forecast_date.date()}")
print(f"Forecast:           {prediction:.2f} cm")

Latest observation: 2026-10-10
Forecast date:      2026-10-17
Forecast:           68.85 cm
